# Modelado del Dataset (2018-2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.  
El objetivo es preparar el dataset final para el entrenamiento del modelo de predicción de severidad en hechos de tránsito, integrando, transformando y seleccionando las variables relevantes obtenidas durante las etapas de EDA y ETL.

In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos limpios

In [2]:
# -- Carga ----------------------------------------------
df_h  = pd.read_csv('../data/clean/hechos_clean.csv')
df_v  = pd.read_csv('../data/clean/vehiculos_involucrados_clean.csv')
df_fl = pd.read_csv('../data/clean/fallecidos_lesionados_clean.csv')

print(f'hechos                : {df_h.shape}')
print(f'vehiculos_involucrados: {df_v.shape}')
print(f'fallecidos_lesionados : {df_fl.shape}')

hechos                : (52488, 18)
vehiculos_involucrados: (80721, 25)
fallecidos_lesionados : (71942, 26)


## 2. Merge de datasets

In [3]:
# -- Merge fallecidos_lesionados + hechos por num_corre ----------------------------------------------
df = df_fl.merge(
    df_h[['num_corre', 'tipo_eve', 'g_hora_5', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu']],
    on='num_corre',
    how='left',
    suffixes=('', '_hecho')
)

print(f'Tras merge con hechos: {df.shape}')
print(f'Nulos nuevos: {df.isnull().sum().sum()}')

Tras merge con hechos: (380434, 31)
Nulos nuevos: 67000


In [4]:
# -- Diagnóstico ----------------------------------------------
print(f'num_corre únicos en hechos         : {df_h["num_corre"].nunique()}')
print(f'num_corre únicos en fallecidos     : {df_fl["num_corre"].nunique()}')
print(f'num_corre únicos en vehiculos      : {df_v["num_corre"].nunique()}')

print(f'\nRegistros en hechos                : {len(df_h)}')
print(f'num_corre duplicados en hechos     : {df_h["num_corre"].duplicated().sum()}')

print(f'\nEjemplos de num_corre duplicados en hechos:')
print(df_h[df_h['num_corre'].duplicated(keep=False)].sort_values('num_corre').head(10))

num_corre únicos en hechos         : 8401
num_corre únicos en fallecidos     : 11274
num_corre únicos en vehiculos      : 13045

Registros en hechos                : 52488
num_corre duplicados en hechos     : 44087

Ejemplos de num_corre duplicados en hechos:
       num_corre  año_ocu  dia_ocu  hora_ocu  g_hora  g_hora_5  mes_ocu  \
0              1     2018        1        16       3         2        1   
35869          1     2023        1        15       3         2        1   
19792          1     2021        1         7       2         1        1   
6395           1     2019        1         5       1         1        1   
44087          1     2024        1        15       3         2        1   
27945          1     2022        1         4       1         1        1   
13442          1     2020        1         1       1         1        1   
35870          2     2023        1        15       3         2        1   
13443          2     2020        1        15       3         2   

In [5]:
# -- Merge correcto usando num_corre + año_carga ----------------------------------------------
df = df_fl.merge(
    df_h[['num_corre', 'año_carga', 'tipo_eve', 'g_hora_5', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu']],
    on=['num_corre', 'año_carga'],
    how='left',
    suffixes=('', '_hecho')
)

print(f'Tras merge con hechos: {df.shape}')
print(f'Nulos nuevos        : {df.isnull().sum().sum()}')

Tras merge con hechos: (71942, 31)
Nulos nuevos        : 97290


In [6]:
# -- Identificar origen de nulos ----------------------------------------------
cols_nuevas = ['tipo_eve_hecho', 'g_hora_5_hecho', 'dia_sem_ocu_hecho', 'mupio_ocu_hecho', 'depto_ocu_hecho']

# verificar si se crearon con sufijo o sin sufijo
print('Columnas añadidas del merge:')
print([c for c in df.columns if c not in df_fl.columns])

print(f'\nRegistros sin match en hechos:')
print(df[df['tipo_eve_hecho'].isnull()]['año_carga'].value_counts().sort_index() if 'tipo_eve_hecho' in df.columns else 'columna no encontrada')

Columnas añadidas del merge:
['tipo_eve_hecho', 'g_hora_5_hecho', 'dia_sem_ocu_hecho', 'mupio_ocu_hecho', 'depto_ocu_hecho']

Registros sin match en hechos:


año_carga
2018    3012
2019    3614
2020    1792
2021    2391
2022    2798
2023    2978
2024    2873
Name: count, dtype: int64


In [7]:
# -- Verificar que df_fl ya tiene todas las columnas necesarias ----------------------------------------------
cols_necesarias = ['tipo_eve', 'g_hora_5', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu']
print('Columnas ya presentes en df_fl:')
for c in cols_necesarias:
    print(f'  {c}: {"✓" if c in df_fl.columns else "✗"}')

# -- Revertir merge y usar df_fl directamente ----------------------------------------------
df = df_fl.copy()
print(f'\nDataset base: {df.shape}')
print(df.columns.tolist())

Columnas ya presentes en df_fl:
  tipo_eve: ✓
  g_hora_5: ✓
  dia_sem_ocu: ✓
  mupio_ocu: ✓
  depto_ocu: ✓

Dataset base: (71942, 26)
['num_corre', 'año_ocu', 'dia_ocu', 'hora_ocu', 'g_hora', 'g_hora_5', 'mes_ocu', 'dia_sem_ocu', 'mupio_ocu', 'depto_ocu', 'zona_ocu', 'sexo_per', 'edad_per', 'g_edad_80ymas', 'g_edad_60ymas', 'edad_quinquenales', 'mayor_menor', 'tipo_veh', 'marca_veh', 'color_veh', 'modelo_veh', 'g_modelo_veh', 'tipo_eve', 'fall_les', 'int_o_noint', 'año_carga']


### ( Dataset base ): df_fl con 71,942 registros × 26 columnas
Todas las variables necesarias ya están en fallecidos_lesionados_clean.
Merge con hechos innecesario — df_fl es el dataset de entrenamiento base.

## 3. Selección de features

In [8]:
# -- Features seleccionadas según EDA ----------------------------------------------
FEATURES = [
    'tipo_eve',        # confiable 0.0% ignorados — predictor más fuerte
    'tipo_veh',        # confiable 3.9% ignorados
    'g_hora_5',        # confiable 0.1% ignorados
    'dia_sem_ocu',     # confiable 0 nulos
    'sexo_per',        # confiable 0.3% ignorados
    'edad_quinquenales', # aceptable 9.7% ignorados
    'mayor_menor',     # confiable 3.0% ignorados
    'depto_ocu',       # confiable 0 nulos
]

TARGET = 'fall_les'

df_model = df[FEATURES + [TARGET]].copy()
print(f'Shape: {df_model.shape}')
print(f'\nDistribución target:')
print(df_model[TARGET].value_counts())

Shape: (71942, 9)

Distribución target:
fall_les
2    58201
1    13741
Name: count, dtype: int64


In [9]:
# -- Verificar códigos ignorados en features seleccionadas ----------------------------------------------
IGNORE_CODES = {
    'tipo_eve'         : 99,
    'tipo_veh'         : 99,
    'g_hora_5'         : 4,
    'dia_sem_ocu'      : None,
    'sexo_per'         : 9,
    'edad_quinquenales': 18,
    'mayor_menor'      : 9,
    'depto_ocu'        : None,
}

print(f'{"Variable":<20} {"Ignorados":>10} {"% del total":>12}')
print('-' * 45)
total = len(df_model)
for col, code in IGNORE_CODES.items():
    if code:
        n = (df_model[col] == code).sum()
        pct = n / total * 100
        print(f'{col:<20} {n:>10,} {pct:>11.1f}%')
    else:
        print(f'{col:<20} {"N/A":>10} {"—":>12}')

Variable              Ignorados  % del total
---------------------------------------------
tipo_eve                    259         0.4%
tipo_veh                  2,824         3.9%
g_hora_5                     39         0.1%
dia_sem_ocu                 N/A            —
sexo_per                    211         0.3%
edad_quinquenales         6,946         9.7%
mayor_menor               2,136         3.0%
depto_ocu                   N/A            —


## 4. Tratamiento de códigos ignorados

El split train/test se realiza aquí, **antes** de imputar, para evitar data leakage: la moda de cada variable se calcula únicamente con `X_train` (excluyendo su propio código "ignorado") y ese mismo valor se aplica igual a `X_train` y a `X_test`.

In [10]:
# -- Split train/test ANTES de imputar (evita data leakage) ----------------------------------------------
from sklearn.model_selection import train_test_split

X = df_model[FEATURES]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -- Moda calculada SOLO con X_train, aplicada igual a X_train y X_test ----------------------------------------------
# tipo_eve   (train)  → imputar con moda
# tipo_veh   (train)  → imputar con moda
# g_hora_5   (train)  → imputar con moda
# sexo_per   (train)  → imputar con moda
# mayor_menor(train)  → imputar con moda
# edad_quinquenales 6946 (9.7%) → mantener como categoría separada

IMPUTAR_MODA = {
    'tipo_eve'   : 99,
    'tipo_veh'   : 99,
    'g_hora_5'   : 4,
    'sexo_per'   : 9,
    'mayor_menor': 9,
}

for col, code in IMPUTAR_MODA.items():
    moda = X_train[X_train[col] != code][col].mode()[0]
    antes_train = (X_train[col] == code).sum()
    antes_test = (X_test[col] == code).sum()
    X_train[col] = X_train[col].replace(code, moda)
    X_test[col] = X_test[col].replace(code, moda)
    print(f'{col:<20} moda(train)={moda}   train: {antes_train} imputados   test: {antes_test} imputados')

tipo_eve             moda(train)=1   train: 200 imputados   test: 59 imputados
tipo_veh             moda(train)=4   train: 2264 imputados   test: 560 imputados
g_hora_5             moda(train)=3   train: 32 imputados   test: 7 imputados
sexo_per             moda(train)=1   train: 166 imputados   test: 45 imputados
mayor_menor          moda(train)=1   train: 1723 imputados   test: 413 imputados


In [11]:
# -- edad_quinquenales — mantener 18 como categoría "Ignorado" ----------------------------------------------
print(f'edad_quinquenales únicos: {sorted(df_model["edad_quinquenales"].unique())}')
print(f'Código 18 (Ignorado)    : {(df_model["edad_quinquenales"] == 18).sum():,} registros')

edad_quinquenales únicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18)]
Código 18 (Ignorado)    : 6,946 registros


In [12]:
# -- edad_quinquenales: mantener 18 como categoría válida ----------------------------------------------
# 9.7% es demasiado para imputar con moda — se conserva como categoría "Ignorado"
# El modelo aprenderá que código 18 = información no disponible

print('edad_quinquenales — se conserva código 18 como categoría separada')
print(f'Distribución final:')
print(df_model['edad_quinquenales'].value_counts().sort_index())

edad_quinquenales — se conserva código 18 como categoría separada
Distribución final:
edad_quinquenales
1      1580
2      2011
3      2321
4      7038
5     12674
6     10384
7      7729
8      5534
9      4085
10     2856
11     2423
12     1728
13     1510
14     1114
15      844
16      555
17      610
18     6946
Name: count, dtype: int64


### edad_quinquenales : 18 categorías incluyendo Ignorado (9.7%)
Grupos 4–6 (15–29 años) concentran el 42% de los registros conocidos.

In [13]:
# -- Verificar estado final del dataset ----------------------------------------------
print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')

print(f'\nCódigos ignorados restantes:')
checks = {'tipo_eve':99, 'tipo_veh':99, 'g_hora_5':4,
          'sexo_per':9, 'mayor_menor':9}
for col, code in checks.items():
    n_train = (X_train[col] == code).sum()
    n_test = (X_test[col] == code).sum()
    print(f'  {col:<20} train={n_train}   test={n_test}')

print(f'\nDistribución target (train):')
print(y_train.value_counts())
print(f'\nDistribución target (test):')
print(y_test.value_counts())

X_train: (57553, 8)
X_test : (14389, 8)

Códigos ignorados restantes:
  tipo_eve             train=0   test=0
  tipo_veh             train=0   test=0
  g_hora_5             train=0   test=0
  sexo_per             train=0   test=0
  mayor_menor          train=0   test=0

Distribución target (train):
fall_les
2    46560
1    10993
Name: count, dtype: int64

Distribución target (test):
fall_les
2    11641
1     2748
Name: count, dtype: int64


### ( Dataset limpio) : 0 códigos ignorados en features críticas

| Variable | Ignorados |
|---|---|
| tipo_eve | 0 |
| tipo_veh | 0 |
| g_hora_5 | 0 |
| sexo_per | 0 |
| mayor_menor | 0 |
| edad_quinquenales | 18 conservado como categoría |

## 5. Encoding de variables categóricas

In [14]:
# -- Todas las features son ordinales codificadas por el INE ----------------------------------------------
# No requieren one-hot encoding — los códigos ya son enteros significativos
# El modelo de árbol los interpretará correctamente como categorías

print('Features y rango de valores (X_train, ya imputado):')
for col in FEATURES:
    print(f'  {col:<22} min={X_train[col].min()}  max={X_train[col].max()}  únicos={X_train[col].nunique()}')

Features y rango de valores (X_train, ya imputado):
  tipo_eve               min=1  max=8  únicos=8
  tipo_veh               min=1  max=24  únicos=20
  g_hora_5               min=1  max=3  únicos=3
  dia_sem_ocu            min=1  max=7  únicos=7
  sexo_per               min=1  max=2  únicos=2
  edad_quinquenales      min=1  max=18  únicos=18
  mayor_menor            min=1  max=2  únicos=2
  depto_ocu              min=1  max=22  únicos=22


### No se requiere encoding adicional

Las variables se conservan codificadas como valores enteros según la clasificación oficial del INE. Esta representación es compatible con los modelos basados en árboles (Random Forest y XGBoost). Para el modelo MLP, las variables son posteriormente estandarizadas mediante StandardScaler, sin requerir codificación one-hot.

## 6. Split train/test

El split ya se realizó en la sección 4 (antes de imputar, para evitar leakage). Aquí se confirma su resultado.

In [15]:
# -- Confirmación del split (ya ejecutado en la sección 4) ----------------------------------------------
print(f'X_train: {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'\nDistribución target en train:')
print(y_train.value_counts(normalize=True).round(3))
print(f'\nDistribución target en test:')
print(y_test.value_counts(normalize=True).round(3))

X_train: (57553, 8)
X_test : (14389, 8)

Distribución target en train:
fall_les
2    0.809
1    0.191
Name: proportion, dtype: float64

Distribución target en test:
fall_les
2    0.809
1    0.191
Name: proportion, dtype: float64


### Split estratificado correcto

| Set | Registros | Fallecido | Lesionado |
|---|---|---|---|
| Train | 57,553 | 19.1% | 80.9% |
| Test | 14,389 | 19.1% | 80.9% |

Proporción target idéntica en train y test — estratificación correcta.

## 6b. Split de validación centralizado (train_final / val)

Se separa aquí, de una sola vez, el split de validación que antes se recalculaba de forma redundante dentro de cada notebook de modelado (06, 07, 08). `train.parquet` y `test.parquet` no se modifican — esta celda solo agrega `train_final.parquet` y `val.parquet` a partir de `X_train`/`y_train`.

In [16]:
# -- Split de validación centralizado (a partir de X_train; no afecta train.parquet/test.parquet) ----------------------------------------------
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

train_final_save = X_train_final.copy()
train_final_save['fall_les'] = y_train_final.values
val_save = X_val.copy()
val_save['fall_les'] = y_val.values

train_final_save.to_parquet('../data/clean/train_final.parquet', index=False)
val_save.to_parquet('../data/clean/val.parquet', index=False)

print(f'✓ train_final.parquet — {train_final_save.shape[0]:,} registros × {train_final_save.shape[1]} columnas')
print(f'✓ val.parquet         — {val_save.shape[0]:,} registros × {val_save.shape[1]} columnas')

✓ train_final.parquet — 43,164 registros × 9 columnas
✓ val.parquet         — 14,389 registros × 9 columnas


## 7. Manejo de desbalance de clases

In [17]:
# -- Verificar desbalance ----------------------------------------------
from collections import Counter

print(f'Distribución original: {Counter(y_train)}')
print(f'Ratio desbalance: {Counter(y_train)[2] / Counter(y_train)[1]:.1f}:1')
print(f'\nEl balanceo se maneja con class_weight="balanced" en el modelo.')
print(f'No se aplica SMOTE para evitar la generación de muestras sintéticas y preservar la distribución original del conjunto de entrenamiento.')

Distribución original: Counter({2: 46560, 1: 10993})
Ratio desbalance: 4.2:1

El balanceo se maneja con class_weight="balanced" en el modelo.
No se aplica SMOTE para evitar la generación de muestras sintéticas y preservar la distribución original del conjunto de entrenamiento.


## 8. Guardar datasets

In [18]:
# -- Guardar train y test ----------------------------------------------
X_train_save = X_train.copy()
X_train_save['fall_les'] = y_train.values
X_test_save = X_test.copy()
X_test_save['fall_les'] = y_test.values

X_train_save.to_parquet('../data/clean/train.parquet', index=False)
X_test_save.to_parquet('../data/clean/test.parquet', index=False)

print(f'✓ train.parquet — {X_train_save.shape[0]:,} registros × {X_train_save.shape[1]} columnas')
print(f'✓ test.parquet  — {X_test_save.shape[0]:,} registros × {X_test_save.shape[1]} columnas')

✓ train.parquet — 57,553 registros × 9 columnas
✓ test.parquet  — 14,389 registros × 9 columnas


### Resumen Dataset Modeling

| Archivo | Registros | Columnas | Tamaño |
|---|---|---|---|
| train.parquet | 57,553 | 9 | — |
| test.parquet | 14,389 | 9 | — |

**Features finales:** tipo_eve, tipo_veh, g_hora_5, dia_sem_ocu,
sexo_per, edad_quinquenales, mayor_menor, depto_ocu

**Target:** fall_les (1=Fallecido, 2=Lesionado)
**Balanceo:** class_weight="balanced" en el modelo — sin SMOTE.
Distribución real conservada en train (19.1% fallecido / 80.9% lesionado).